In [1]:
#  Question 1: Build an End-to-End RAG + Agent System (25 Marks)
# 🧩 Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status
# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers
# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM
# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used
# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation
# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality
# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality

In [ ]:
# Load Docs → Split → Embed → Store → Retrieve → Generate → Answer
                           # ↑
                        # Agent decides

In [3]:
# !pip install langchain faiss-cpu sentence-transformers transformers

In [1]:
from langchain_community.document_loaders import TextLoader

docs = []
files = ["course_policy.txt", "lecture_notes.txt", "assignments.txt", "faq.txt"]

for file in files:
    loader = TextLoader(file)
    docs.extend(loader.load())

In [30]:
with open("course_policy.txt", "r") as f:
    content = f.read()

print(content)

1. Refund Policy:
Students can request a full refund within 7 days of enrollment.
After 7 days, no refund will be issued.

2. Course Deadline:
All courses must be completed within 6 months of enrollment.

3. Attendance Policy:
Students must complete at least 70% of lectures to receive certification.

4. Assignment Policy:
Assignments must be submitted before the deadline.
Late submissions will receive a penalty of 10% per day.


In [29]:
with open("lecture_notes.txt", "r") as f:
    content = f.read()

print(content)

Lecture 1: Introduction to AI
Artificial Intelligence is the simulation of human intelligence in machines.

Lecture 2: Machine Learning
Machine Learning is a subset of AI that enables systems to learn from data.

Lecture 3: Deep Learning
Deep Learning uses neural networks with multiple layers.

Lecture 4: NLP
Natural Language Processing helps machines understand human language.


In [31]:
with open("assignments.txt", "r") as f:
    content = f.read()

print(content)

Assignment 1:
Topic: Basics of AI
Deadline: 10 October 2026

Assignment 2:
Topic: Machine Learning Models
Deadline: 25 October 2026

Assignment 3:
Topic: Neural Networks
Deadline: 5 November 2026


In [32]:
with open("faq.txt", "r") as f:
    content = f.read()

print(content)

Q: How can I enroll in a course?
A: You can enroll through the dashboard.

Q: Can I get a certificate?
A: Yes, after completing the course and assignments.

Q: What happens if I miss a deadline?
A: A penalty may be applied.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\naina\AppData\Local\Temp\ipykernel_18244\2985419238.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embedding)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

In [17]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [18]:
def generate_answer(context, question):

    prompt = f"""
    Answer the question using ONLY the context below.
    If not found, say 'Not found'.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    result = generator(
        prompt,
        max_new_tokens=100,
        truncation=True
    )
    
    return result[0]['generated_text']

In [19]:
students_db = {
    "ST101": {"name": "Rahul", "progress": "80%", "status": "Completed"},
    "ST102": {"name": "Priya", "progress": "60%", "status": "In Progress"}
}

def get_student_status(student_id):
    return students_db.get(student_id, "Student not found")

In [20]:
def agent_router(query):
    
    keywords = ["my", "progress", "status", "enrollment"]
    
    for word in keywords:
        if word in query.lower():
            return "tool"
    
    return "rag"

In [21]:
def assistant(query, student_id="ST101"):
    
    decision = agent_router(query)
    
    # TOOL CASE
    if decision == "tool":
        return get_student_status(student_id)
    
    # RAG CASE
    else:
        docs = retriever.invoke(query)   # ✅ FIX HERE
        
        if not docs:
            return "No relevant information found."
        
        context = "\n".join([doc.page_content for doc in docs])
        
        return generate_answer(context, query)

In [37]:
print(assistant("What is the refund policy?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer the question using ONLY the context below.
    If not found, say 'Not found'.

    Context:
    1. Refund Policy:
Students can request a full refund within 7 days of enrollment.
After 7 days, no refund will be issued.

2. Course Deadline:
All courses must be completed within 6 months of enrollment.

3. Attendance Policy:
Students must complete at least 70% of lectures to receive certification.
4. Assignment Policy:
Assignments must be submitted before the deadline.
Late submissions will receive a penalty of 10% per day.
Q: How can I enroll in a course?
A: You can enroll through the dashboard.

Q: Can I get a certificate?
A: Yes, after completing the course and assignments.

Q: What happens if I miss a deadline?
A: A penalty may be applied.

    Question:
    What is the refund policy?

    Answer:
     Answer the question using ONLY the context below.
Q: What does the refund policy mean?
A: None.
Q: Can I use a certificate?
A: Yes, after completing the course and assignment

In [34]:
print(assistant("How can I enroll in a course?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer the question using ONLY the context below.
    If not found, say 'Not found'.

    Context:
    Q: How can I enroll in a course?
A: You can enroll through the dashboard.

Q: Can I get a certificate?
A: Yes, after completing the course and assignments.

Q: What happens if I miss a deadline?
A: A penalty may be applied.
1. Refund Policy:
Students can request a full refund within 7 days of enrollment.
After 7 days, no refund will be issued.

2. Course Deadline:
All courses must be completed within 6 months of enrollment.

3. Attendance Policy:
Students must complete at least 70% of lectures to receive certification.
4. Assignment Policy:
Assignments must be submitted before the deadline.
Late submissions will receive a penalty of 10% per day.

    Question:
    How can I enroll in a course?

    Answer:
     If not found, say 'Not found'.
A: You can enroll through the dashboard.
Q: Can I get a certificate?
A: You can enroll through the dashboard.
Q: Can I get a certificate?
A:

In [36]:
print(assistant("What is my progress?", "ST102"))

{'name': 'Priya', 'progress': '60%', 'status': 'In Progress'}
